# Fundamentals-Based Screening and Ranking of Private Companies in Colombia

Full methodology, data limitations, three bugs found and fixed during implementation, and
results are documented in **[README.md](README.md)** — this notebook is the pipeline itself.
Source: Colombia's Superintendencia de Sociedades (Supersociedades) IFRS-based XBRL filings.

## 1. Load and translate raw filings

Six Excel exports: the balance sheet, the income statement, and four supporting note tables
(revenue, payables, PP&E, intangibles) that back out the detail behind certain balance-sheet
line items.

In [ ]:
import pandas as pd
import numpy as np
import os

base_path = r"C:\Users\secar\OneDrive\Documents\UCLA\research"

balance_path = os.path.join(base_path, "210030_Estado de situación financiera, corriente_no corriente.xlsx")
income_path = os.path.join(base_path, "310030_Estado de resultado integral, resultado del periodo, por funcion de gasto.xlsx")
revenue_notes_path = os.path.join(base_path, "801800a_Notas - Análisis de Ingresos.xlsx")
payables_notes_path = os.path.join(base_path, "801600_Notas - Cuentas comerciales por pagar, otras cuentas por pagar y otros pasivos.xlsx")
ppe_notes_path = os.path.join(base_path, "801200_Notas - Propiedades, planta y equipo.xlsx")
intangibles_notes_path = os.path.join(base_path, "801300_Notas - Activos intangibles distintos de la plusvalía.xlsx")

balance = pd.read_excel(balance_path)
income = pd.read_excel(income_path)
revenue_notes = pd.read_excel(revenue_notes_path)
payables_notes = pd.read_excel(payables_notes_path)
ppe_notes = pd.read_excel(ppe_notes_path)
intangibles_notes = pd.read_excel(intangibles_notes_path)

print("files successfully loaded")

## 2. Deduplicate to each company's latest reporting period

**Fix #1** (see README §"Data structure issues"): a few NITs (company tax IDs) have more than
one row for the "current period" — keep each company's *latest* `Fecha de Corte`, not just the
first row encountered.

In [ ]:
def clean_nit(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip()

for df in [balance, income, revenue_notes, payables_notes, ppe_notes, intangibles_notes]:
    if "NIT" in df.columns:
        df["NIT"] = df["NIT"].apply(clean_nit)


def latest_period(df, date_col="Fecha de Corte"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    idx = df.groupby("NIT")[date_col].idxmax()
    return df.loc[idx]


balance = balance[balance["Periodo"].astype(str).str.strip().eq("Periodo Actual")].copy()
balance = latest_period(balance)

income = income[income["Periodo"].astype(str).str.strip().eq("Periodo Actual")].copy()
income = latest_period(income)

assert balance["NIT"].is_unique, "balance still has duplicate NITs after latest_period"
assert income["NIT"].is_unique, "income still has duplicate NITs after latest_period"

## 3. Select and translate balance sheet / income statement fields

In [ ]:
balance_cols = [
    "NIT", "Razón social de la sociedad",
    "Clasificación Industrial Internacional Uniforme Versión 4 A.C (CIIU)",
    "Fecha de Corte",
    "Activos corrientes totales (CurrentAssets)",
    "Total de activos (Assets)",
    "Pasivos corrientes totales (CurrentLiabilities)",
    "Total pasivos (Liabilities)",
    "Patrimonio total (Equity)",
    "Propiedades, planta y equipo (PropertyPlantAndEquipment)",
    "Activos intangibles distintos de la plusvalía (IntangibleAssetsOtherThanGoodwill)",
    "Efectivo y equivalentes al efectivo (CashAndCashEquivalents)",
    "Inventarios corrientes (Inventories)",
    "Cuentas comerciales por cobrar y otras cuentas por cobrar corrientes (TradeAndOtherCurrentReceivables)",
    "Otros pasivos financieros corrientes (OtherCurrentFinancialLiabilities)",
    "Otros pasivos financieros no corrientes (OtherNoncurrentFinancialLiabilities)"
]

balance_base = balance[[c for c in balance_cols if c in balance.columns]].copy()
balance_base = balance_base.rename(columns={
    "Razón social de la sociedad": "company_name",
    "Clasificación Industrial Internacional Uniforme Versión 4 A.C (CIIU)": "sector_ciiu",
    "Fecha de Corte": "report_date",
    "Activos corrientes totales (CurrentAssets)": "current_assets",
    "Total de activos (Assets)": "total_assets",
    "Pasivos corrientes totales (CurrentLiabilities)": "current_liabilities",
    "Total pasivos (Liabilities)": "total_liabilities",
    "Patrimonio total (Equity)": "equity",
    "Propiedades, planta y equipo (PropertyPlantAndEquipment)": "ppe_total_bs",
    "Activos intangibles distintos de la plusvalía (IntangibleAssetsOtherThanGoodwill)": "intangibles_total_bs",
    "Efectivo y equivalentes al efectivo (CashAndCashEquivalents)": "cash",
    "Inventarios corrientes (Inventories)": "inventory",
    "Cuentas comerciales por cobrar y otras cuentas por cobrar corrientes (TradeAndOtherCurrentReceivables)": "receivables",
    "Otros pasivos financieros corrientes (OtherCurrentFinancialLiabilities)": "financial_liabilities_current",
    "Otros pasivos financieros no corrientes (OtherNoncurrentFinancialLiabilities)": "financial_liabilities_noncurrent"
})

balance_base["financial_liabilities_current"] = balance_base.get("financial_liabilities_current", np.nan)
balance_base["financial_liabilities_noncurrent"] = balance_base.get("financial_liabilities_noncurrent", np.nan)
balance_base["financial_debt"] = (
    pd.to_numeric(balance_base["financial_liabilities_current"], errors="coerce").fillna(0)
    + pd.to_numeric(balance_base["financial_liabilities_noncurrent"], errors="coerce").fillna(0)
)

income_cols = [
    "NIT",
    "Ingresos de actividades ordinarias (Revenue)",
    "Costo de ventas (CostOfSales)",
    "Ganancia bruta (GrossProfit)",
    "Gastos de administración (AdministrativeExpense)",
    "Gastos de ventas (DistributionCosts)",
    "Ganancia (pérdida) por actividades de operación (ProfitLossFromOperatingActivities)",
    "Ingresos financieros (FinanceIncome)",
    "Costos financieros (FinanceCosts)",
    "Ganancia (pérdida), antes de impuestos (ProfitLossBeforeTax)",
    "Ganancia (pérdida) (ProfitLoss)"
]
income_base = income[[c for c in income_cols if c in income.columns]].copy()
income_base = income_base.rename(columns={
    "Ingresos de actividades ordinarias (Revenue)": "revenue",
    "Costo de ventas (CostOfSales)": "cost_of_sales",
    "Ganancia bruta (GrossProfit)": "gross_profit",
    "Gastos de administración (AdministrativeExpense)": "admin_expense",
    "Gastos de ventas (DistributionCosts)": "selling_expense",
    "Ganancia (pérdida) por actividades de operación (ProfitLossFromOperatingActivities)": "operating_profit",
    "Ingresos financieros (FinanceIncome)": "finance_income",
    "Costos financieros (FinanceCosts)": "finance_costs",
    "Ganancia (pérdida), antes de impuestos (ProfitLossBeforeTax)": "profit_before_tax",
    "Ganancia (pérdida) (ProfitLoss)": "net_income"
})
income_base["ebitda_proxy"] = pd.to_numeric(income_base["operating_profit"], errors="coerce")

## 4. Aggregate the note-level rollforward tables — correctly

**Fix #2** (see README): filter each note table to the single correct `Concepto` row and match
on NIT + latest `Fecha de Corte`, instead of a blind `max()`/`sum()` across every rollforward
line item and every quarter.

In [ ]:
# --- PP&E notes: keep only the end-of-period balance ---
PPE_COL = "Propiedades, planta y equipo [miembro] (PropertyPlantAndEquipmentMember)"
ppe_end = ppe_notes[ppe_notes["Concepto"].str.strip()
                     .eq("Propiedades, planta y equipo al final del periodo")].copy()
ppe_end[PPE_COL] = pd.to_numeric(ppe_end[PPE_COL], errors="coerce")
ppe_end = latest_period(ppe_end)
ppe_notes_agg = ppe_end[["NIT", PPE_COL]].rename(columns={PPE_COL: "ppe_note_total"})

# --- Intangibles notes: same pattern ---
INT_COL = "Activos intangibles distintos de la plusvalía [miembro] (IntangibleAssetsOtherThanGoodwillMember)"
int_end = intangibles_notes[intangibles_notes["Concepto"].str.strip()
                             .eq("Activos intangibles distintos de la plusvalía al final del periodo")].copy()
int_end[INT_COL] = pd.to_numeric(int_end[INT_COL], errors="coerce")
int_end = latest_period(int_end)
intangibles_notes_agg = int_end[["NIT", INT_COL]].rename(columns={INT_COL: "intangibles_note_total"})

# --- Revenue notes: keep only the grand total, not a sub-category ---
REV_COL = "Saldo (SaldoIngresoGastos)"
rev_total = revenue_notes[revenue_notes["Concepto"].str.strip()
                           .eq("Total de Ingresos de actividades ordinarias")].copy()
rev_total[REV_COL] = pd.to_numeric(rev_total[REV_COL], errors="coerce")
rev_total = latest_period(rev_total)
revenue_notes_agg = rev_total[["NIT", REV_COL]].rename(columns={REV_COL: "revenue_note_total"})

# --- Payables notes: sum ONLY current-total + noncurrent-total, not their sub-lines ---
TOTAL_COL = "Total saldo del ejercicio [miembro] (CxPTotalMiembro)"
PASTDUE_COL = "Total obligaciones vencidas [miembro] (CxPTotalObligacionesVencidasMiembro)"
keep_concepts = [
    "Total cuentas comerciales por pagar, otras cuentas por pagar y otros pasivos corrientes",
    "Total cuentas comerciales por pagar, otras cuentas por pagar y otros pasivos NO corrientes",
]
pay_tot = payables_notes[payables_notes["Concepto"].str.strip().isin(keep_concepts)].copy()
pay_tot[TOTAL_COL] = pd.to_numeric(pay_tot[TOTAL_COL], errors="coerce")
pay_tot[PASTDUE_COL] = pd.to_numeric(pay_tot[PASTDUE_COL], errors="coerce")
pay_tot["Fecha de Corte"] = pd.to_datetime(pay_tot["Fecha de Corte"])
latest_date = pay_tot.groupby("NIT")["Fecha de Corte"].transform("max")
pay_tot = pay_tot[pay_tot["Fecha de Corte"].eq(latest_date)]
pay_tot = pay_tot.groupby("NIT", as_index=False)[[TOTAL_COL, PASTDUE_COL]].sum(min_count=1)
payables_notes_agg = pay_tot.rename(columns={
    TOTAL_COL: "payables_total_note",
    PASTDUE_COL: "past_due_payables_note"
})

df_base = (
    balance_base
    .merge(income_base, on="NIT", how="left")
    .merge(revenue_notes_agg, on="NIT", how="left")
    .merge(payables_notes_agg, on="NIT", how="left")
    .merge(ppe_notes_agg, on="NIT", how="left")
    .merge(intangibles_notes_agg, on="NIT", how="left")
)

assert len(df_base) == df_base["NIT"].nunique(), \
    f"merge produced duplicate NITs: {len(df_base)} rows vs {df_base['NIT'].nunique()} unique NITs"

numeric_cols = [
    "current_assets", "total_assets", "current_liabilities", "total_liabilities", "equity",
    "ppe_total_bs", "intangibles_total_bs", "cash", "inventory", "receivables",
    "financial_liabilities_current", "financial_liabilities_noncurrent", "financial_debt",
    "revenue", "cost_of_sales", "gross_profit", "admin_expense", "selling_expense",
    "operating_profit", "finance_income", "finance_costs", "profit_before_tax",
    "net_income", "ebitda_proxy", "revenue_note_total", "payables_total_note",
    "past_due_payables_note", "ppe_note_total", "intangibles_note_total"
]
for c in numeric_cols:
    if c in df_base.columns:
        df_base[c] = pd.to_numeric(df_base[c], errors="coerce")

print("rows:", len(df_base))
print("unique NITs:", df_base["NIT"].nunique())
df_base.head()

## 5. Build financial ratios

Combine note-level totals with balance-sheet totals (preferring the note when available), then
compute the ratios that feed the scoring models: liquidity, leverage, profitability, and asset
composition. Ratios are winsorized at the 1st/99th percentile to limit outlier distortion.

In [ ]:
df_factors = df_base.copy()

# use note totals when available, otherwise fall back to balance sheet totals
df_factors["ppe_total"] = df_factors["ppe_note_total"].combine_first(df_factors["ppe_total_bs"]).fillna(0)
df_factors["intangibles_total"] = df_factors["intangibles_note_total"].combine_first(df_factors["intangibles_total_bs"]).fillna(0)

# if revenue note exists and is more complete, keep it as a comparison check
df_factors["revenue_final"] = df_factors["revenue"].combine_first(df_factors["revenue_note_total"])

df_factors["current_ratio"] = df_factors["current_assets"] / df_factors["current_liabilities"].replace(0, np.nan)
df_factors["cash_ratio"] = df_factors["cash"] / df_factors["current_liabilities"].replace(0, np.nan)
df_factors["working_capital"] = df_factors["current_assets"] - df_factors["current_liabilities"]

df_factors["debt_to_assets"] = df_factors["financial_debt"] / df_factors["total_assets"].replace(0, np.nan)
df_factors["liabilities_to_assets"] = df_factors["total_liabilities"] / df_factors["total_assets"].replace(0, np.nan)
df_factors["debt_to_equity"] = df_factors["financial_debt"] / df_factors["equity"].replace(0, np.nan)

df_factors["gross_margin"] = df_factors["gross_profit"] / df_factors["revenue_final"].replace(0, np.nan)
df_factors["operating_margin"] = df_factors["operating_profit"] / df_factors["revenue_final"].replace(0, np.nan)
df_factors["net_margin"] = df_factors["net_income"] / df_factors["revenue_final"].replace(0, np.nan)

df_factors["roa"] = df_factors["net_income"] / df_factors["total_assets"].replace(0, np.nan)
df_factors["roe"] = df_factors["net_income"] / df_factors["equity"].replace(0, np.nan)
df_factors["ebitda_to_assets"] = df_factors["ebitda_proxy"] / df_factors["total_assets"].replace(0, np.nan)

df_factors["ppe_to_assets"] = df_factors["ppe_total"] / df_factors["total_assets"].replace(0, np.nan)
df_factors["intangibles_to_assets"] = df_factors["intangibles_total"] / df_factors["total_assets"].replace(0, np.nan)
df_factors["inventory_to_assets"] = df_factors["inventory"] / df_factors["total_assets"].replace(0, np.nan)
df_factors["receivables_to_assets"] = df_factors["receivables"] / df_factors["total_assets"].replace(0, np.nan)

df_factors["payables_past_due_share"] = (
    df_factors["past_due_payables_note"] / df_factors["payables_total_note"].replace(0, np.nan)
)
df_factors["finance_cost_burden"] = df_factors["finance_costs"] / df_factors["revenue_final"].replace(0, np.nan)
df_factors["interest_coverage_proxy"] = df_factors["operating_profit"] / df_factors["finance_costs"].replace(0, np.nan)
df_factors["log_assets"] = np.log(df_factors["total_assets"].where(df_factors["total_assets"] > 0))

ratio_cols = [
    "current_ratio", "cash_ratio", "debt_to_assets", "liabilities_to_assets", "debt_to_equity",
    "gross_margin", "operating_margin", "net_margin", "roa", "roe", "ebitda_to_assets",
    "ppe_to_assets", "intangibles_to_assets", "inventory_to_assets", "receivables_to_assets",
    "payables_past_due_share", "finance_cost_burden", "interest_coverage_proxy"
]
for c in ratio_cols:
    df_factors[c] = df_factors[c].replace([np.inf, -np.inf], np.nan)
    lo, hi = df_factors[c].quantile(0.01), df_factors[c].quantile(0.99)
    df_factors[c] = df_factors[c].clip(lower=lo, upper=hi)

df_factors[[
    "NIT", "company_name", "sector_ciiu", "revenue_final", "total_assets",
    "current_ratio", "debt_to_assets", "operating_margin", "roa",
    "ppe_to_assets", "payables_past_due_share", "interest_coverage_proxy"
]].head()

## 6. Reference scoring system: quality / safety / value

An equal-weighted top-30 reference ranking, built from z-scored ratios with the disclosed
weights in the README.

In [ ]:
def zscore(series):
    s = pd.to_numeric(series, errors="coerce")
    mu, sd = s.mean(), s.std()
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.nan, index=s.index)
    return (s - mu) / sd

# positive = better
df_factors["z_operating_margin"] = zscore(df_factors["operating_margin"])
df_factors["z_roa"] = zscore(df_factors["roa"])
df_factors["z_current_ratio"] = zscore(df_factors["current_ratio"])
df_factors["z_interest_coverage"] = zscore(df_factors["interest_coverage_proxy"])
# negative = worse, so flip sign
df_factors["z_debt_to_assets_neg"] = -zscore(df_factors["debt_to_assets"])
df_factors["z_past_due_payables_neg"] = -zscore(df_factors["payables_past_due_share"])
df_factors["z_finance_cost_burden_neg"] = -zscore(df_factors["finance_cost_burden"])
df_factors["z_ppe_to_assets"] = zscore(df_factors["ppe_to_assets"])

df_factors["quality_score"] = (
    0.25 * df_factors["z_operating_margin"] + 0.20 * df_factors["z_roa"] +
    0.15 * df_factors["z_current_ratio"] + 0.15 * df_factors["z_interest_coverage"] +
    0.15 * df_factors["z_debt_to_assets_neg"] + 0.10 * df_factors["z_past_due_payables_neg"]
)
df_factors["safety_score"] = (
    0.35 * df_factors["z_debt_to_assets_neg"] + 0.25 * df_factors["z_past_due_payables_neg"] +
    0.20 * df_factors["z_finance_cost_burden_neg"] + 0.20 * df_factors["z_current_ratio"]
)
df_factors["value_score"] = 0.50 * df_factors["z_ppe_to_assets"] + 0.50 * df_factors["z_roa"]
df_factors["total_score"] = (
    0.50 * df_factors["quality_score"] + 0.30 * df_factors["safety_score"] + 0.20 * df_factors["value_score"]
)
df_factors["portfolio_rank"] = df_factors["total_score"].rank(ascending=False, method="dense")

top_companies = df_factors.sort_values("total_score", ascending=False)
top_companies[[
    "portfolio_rank", "NIT", "company_name", "sector_ciiu",
    "quality_score", "safety_score", "value_score", "total_score"
]].head(20)

## 7. Candidate portfolio: fundamental-strength vs. risk optimization

The second, independent scoring system — `mu_proxy` (fundamental strength) minus a risk-aversion
multiple of `risk_proxy` — used to actually select and weight the candidate portfolio, capped at
8% per company and 25% per fine-grained CIIU sector.

In [ ]:
def zscore2(s):
    s = s.astype(float)
    return (s - s.mean()) / s.std()

df_opt = df_factors.copy()

df_opt["z_operating_margin"] = zscore2(df_opt["operating_margin"])
df_opt["z_roa"] = zscore2(df_opt["roa"])
df_opt["z_current_ratio"] = zscore2(df_opt["current_ratio"])
df_opt["z_ppe_to_assets"] = zscore2(df_opt["ppe_to_assets"])
df_opt["z_interest_coverage"] = zscore2(df_opt["interest_coverage_proxy"])

# fundamental strength proxy -- more profitable, liquid, asset-backed, better debt coverage
df_opt["mu_proxy"] = (
    0.30 * df_opt["z_operating_margin"] + 0.25 * df_opt["z_roa"] +
    0.15 * df_opt["z_current_ratio"] + 0.15 * df_opt["z_ppe_to_assets"] +
    0.15 * df_opt["z_interest_coverage"]
)

df_opt["z_debt_to_assets"] = zscore2(df_opt["debt_to_assets"])
df_opt["z_payables_risk"] = zscore2(df_opt["payables_past_due_share"])
df_opt["z_financing_cost"] = zscore2(df_opt["finance_cost_burden"])

df_opt["risk_proxy"] = (
    0.50 * df_opt["z_debt_to_assets"] + 0.30 * df_opt["z_payables_risk"] + 0.20 * df_opt["z_financing_cost"]
)

risk_aversion = 0.7
df_opt["score_net"] = df_opt["mu_proxy"] - risk_aversion * df_opt["risk_proxy"]

# full-universe snapshot for factor/sector diagnostics, taken BEFORE any top-N filtering
# -- this separation is the fix for the selection-circularity issue (see README)
df_full = df_opt.copy()

In [ ]:
df_opt = df_opt.sort_values("score_net", ascending=False)
df_opt = df_opt[df_opt["score_net"] > 0].copy()   # keep only positive expected contribution

top_n = 50
df_opt = df_opt.head(top_n)
df_opt["raw_weight"] = df_opt["score_net"].clip(lower=0)
df_opt["weight_prelim"] = df_opt["raw_weight"] / df_opt["raw_weight"].sum()

max_company_weight = 0.08
df_opt["weight_capped"] = df_opt["weight_prelim"].clip(upper=max_company_weight)
df_opt["weight_final"] = df_opt["weight_capped"] / df_opt["weight_capped"].sum()

### Apply the 25% fine-grained sector cap

Walk companies in score order, capping each CIIU sector's cumulative weight at 25% and
redistributing whatever a capped company can't take.

In [ ]:
max_sector_weight = 0.25
df_opt = df_opt.sort_values("score_net", ascending=False)

sector_running = {}
weights = []
for i, row in df_opt.iterrows():
    sector = row["sector_ciiu"]
    proposed_weight = row["weight_final"]
    used_weight = sector_running.get(sector, 0)
    allowable = max_sector_weight - used_weight
    final_weight = max(0, min(proposed_weight, allowable))
    weights.append(final_weight)
    sector_running[sector] = used_weight + final_weight

df_opt["weight_sector_adj"] = weights
df_opt["weight_sector_adj"] = df_opt["weight_sector_adj"] / df_opt["weight_sector_adj"].sum()

portfolio_final = df_opt[[
    "NIT", "company_name", "sector_ciiu", "mu_proxy", "risk_proxy", "score_net", "weight_sector_adj"
]].copy()
portfolio_final = portfolio_final.sort_values("weight_sector_adj", ascending=False)
portfolio_final.head(25)

## 8. Broad-sector (macro-bucket) allocation

A separate, coarser lens: map every company's CIIU code to one of nine broad economic buckets,
score each bucket by reliability-adjusted average `score_net`, and floor negative strength at 0
before normalizing — **the fix for the sign-flip bug** described in the README (an unclipped
negative denominator previously inverted the entire ranking).

In [ ]:
def sector_bucket(code):
    code = str(code)
    if code.startswith("L"):
        return "real_estate"
    if code.startswith("A"):
        return "agriculture"
    if code.startswith("B"):
        return "energy_mining"
    if code.startswith("C"):
        return "manufacturing"
    if code.startswith("G"):
        return "trade"
    if code.startswith("M"):
        return "professional_services"
    if code.startswith("J"):
        return "technology"
    if code.startswith("F"):
        return "construction"
    return "other"


df_full["sector_group"] = df_full["sector_ciiu"].apply(sector_bucket)

macro_sector_weights = (
    df_full.groupby("sector_group").agg(score=("score_net", "mean"), n=("NIT", "count"))
)
macro_sector_weights["strength"] = macro_sector_weights["score"] * np.sqrt(macro_sector_weights["n"])

# floor negative strength at 0 before normalizing -- otherwise a negative
# denominator flips every sign and inverts the ranking (worst sectors get
# the largest weight, best sectors go negative)
macro_sector_weights["strength"] = macro_sector_weights["strength"].clip(lower=0)
macro_sector_weights["weight"] = macro_sector_weights["strength"] / macro_sector_weights["strength"].sum()

macro_sector_weights.sort_values("weight", ascending=False)

**Result:** Professional Services (53.1%), Manufacturing (20.9%), Real Estate (19.6%), and
Technology (6.4%) capture the entire positive-scoring universe; five buckets — including Trade,
the largest by company count at 834 firms — average a negative score and correctly receive zero
weight. Full company-level results, the disclosed factor weights, and the economic interpretation
of these exposures are in `README.md`.